# Heston × QuantLib real-market cross-check — temporary workaround

This notebook is a temporary QuantLib reassurance check while the library does not yet expose a plug-and-play Heston-ready quote preparation workflow.

Temporary workaround:
- load saved model-validation bundles from `data/gold/model_validation_bundle`;
- exclude the old invalid `SPY / 2026-06-03` bundle;
- build a deterministic Heston-calibration-ready quote universe inside the notebook;
- calibrate this library's Heston implementation;
- price the same selected quotes with this library and QuantLib using the same fitted parameters.

This notebook should eventually be simplified once the library exposes an official bundle → model-ready workflow.


In [1]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get(
    "OPL_PROJECT_ROOT",
    r"C:\Users\ouwez\Documents\Quant\option-pricing-library",
)).expanduser()

DATA_ROOT = Path(os.environ.get("OPL_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser()
MODEL_VALIDATION_ROOT = DATA_ROOT / "gold" / "model_validation_bundle"

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

UNDERLYINGS = ["SPY", "QQQ", "IWM", "MSFT", "NVDA", "TSLA"]
EXCLUDE_BUNDLES = {("SPY", "2026-06-03")}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_VALIDATION_ROOT:", MODEL_VALIDATION_ROOT)


PROJECT_ROOT: C:\Users\ouwez\Documents\Quant\option-pricing-library
MODEL_VALIDATION_ROOT: C:\Users\ouwez\Documents\Quant\option-pricing-library\data\gold\model_validation_bundle


In [2]:
from option_pricing.diagnostics.heston import (
    heston_calibration_fit_summary,
    run_heston_calibration_fit_diagnostics,
)
from option_pricing.marketdata.gold import (
    heston_quote_set_from_frame,
    market_data_snapshot_from_json,
)
from option_pricing.models.heston import HestonParams
from option_pricing.models.heston.calibration import (
    HestonCalibrationBounds,
    calibrate_heston_multistart,
    preflight_heston_quotes,
)
from option_pricing.numerics.quadrature import QuadratureConfig
from option_pricing.pricers.heston import heston_price_from_ctx
from option_pricing.types import OptionType

try:
    import QuantLib as ql
    QUANTLIB_AVAILABLE = True
    print("QuantLib available:", getattr(ql, "__version__", "unknown version"))
except Exception as exc:
    ql = None
    QUANTLIB_AVAILABLE = False
    print("QuantLib unavailable:", type(exc).__name__, exc)


QuantLib available: 1.41


In [3]:
# Calibration/pricing settings.
OBJECTIVE_TYPE = "price_rmse"
MAX_SEEDS = 8
MAX_NFEV = 300
USE_ANALYTIC_JAC = True

ROBUST_QUAD_CFG = QuadratureConfig(u_max=260.0, n_panels=56, nodes_per_panel=32)
QL_ENGINE_ORDER = 192
ROUND_LIBRARY_TAU_TO_QL_DAYS = True

IMPLEMENTATION_RMSE_TOL = 2e-3
IMPLEMENTATION_MAX_ABS_TOL = 5e-3

# Temporary deterministic Heston-ready universe filter.
MIN_QUOTES = 80
MIN_EXPIRIES = 3
MIN_QUOTES_PER_RIGHT = 20
MIN_EXPIRY_DAYS = 7.0
MAX_EXPIRY_DAYS = 180.0
MAX_ABS_LOG_MONEYNESS = 0.35
MAX_RELATIVE_SPREAD = 0.35
MAX_QUOTES_TOTAL = 900
MAX_QUOTES_PER_EXPIRY_RIGHT = 80


In [4]:
@dataclass(frozen=True)
class Bundle:
    root: Path
    underlying: str
    date: str
    run_id: str


def _part(path: Path, name: str) -> str:
    prefix = f"{name}="
    for p in path.parts:
        if p.startswith(prefix):
            return p.split("=", 1)[1]
    return ""


bundle_paths = sorted(
    p for p in MODEL_VALIDATION_ROOT.glob("underlying=*/date=*/run_id=*")
    if (p / "market_data.json").exists()
    and (p / "heston_quotes.parquet").exists()
    and (p / "manifest.json").exists()
)

bundles = []
for p in bundle_paths:
    underlying = _part(p, "underlying")
    date = _part(p, "date")
    run_id = _part(p, "run_id")
    if UNDERLYINGS and underlying not in UNDERLYINGS:
        continue
    if (underlying, date) in EXCLUDE_BUNDLES:
        continue
    bundles.append(Bundle(root=p, underlying=underlying, date=date, run_id=run_id))

print(f"bundles found: {len(bundles)}")
for b in bundles:
    print(b.underlying, b.date, b.run_id)


bundles found: 6
IWM 2026-06-06 b4a-20260606T111054Z-d64ab9a6
MSFT 2026-06-06 b4a-20260606T111126Z-0d4d8cc7
NVDA 2026-06-06 b4a-20260606T111142Z-82961440
QQQ 2026-06-06 b4a-20260606T111029Z-2c8a4d1d
SPY 2026-06-06 b4a-20260606T111002Z-a037b8fe
TSLA 2026-06-06 b4a-20260606T111158Z-a2346800


In [5]:
def load_bundle(bundle: Bundle) -> dict[str, Any]:
    market_payload = json.loads((bundle.root / "market_data.json").read_text(encoding="utf-8"))
    market_snapshot = market_data_snapshot_from_json(market_payload)
    heston_quotes = pd.read_parquet(bundle.root / "heston_quotes.parquet")
    return {
        "market_payload": market_payload,
        "market_snapshot": market_snapshot,
        "market_data": market_snapshot.market_data,
        "heston_quotes": heston_quotes,
    }


def add_moneyness_columns(frame: pd.DataFrame, market_data) -> pd.DataFrame:
    out = frame.copy().reset_index(drop=True)
    ctx = market_data.to_context()
    expiry = pd.to_numeric(out["expiry_years"], errors="coerce").to_numpy(dtype=float)
    strike = pd.to_numeric(out["strike"], errors="coerce").to_numpy(dtype=float)
    forward = np.asarray([ctx.fwd(float(t)) if np.isfinite(t) and t > 0 else np.nan for t in expiry])
    mid = pd.to_numeric(out["mid"], errors="coerce")
    bid = pd.to_numeric(out["bid"], errors="coerce")
    ask = pd.to_numeric(out["ask"], errors="coerce")

    out["forward"] = forward
    out["expiry_days"] = expiry * 365.0
    out["log_moneyness"] = np.log(strike / forward)
    out["abs_log_moneyness"] = np.abs(out["log_moneyness"])
    out["relative_spread"] = (ask - bid) / mid.replace(0.0, np.nan)
    return out


def prepare_heston_ready_frame(frame: pd.DataFrame, market_data) -> tuple[pd.DataFrame, pd.DataFrame]:
    enriched = add_moneyness_columns(frame, market_data)

    iv = pd.to_numeric(enriched["iv"], errors="coerce")
    vega = pd.to_numeric(enriched["vega"], errors="coerce")
    expiry = pd.to_numeric(enriched["expiry_years"], errors="coerce")
    strike = pd.to_numeric(enriched["strike"], errors="coerce")
    mid = pd.to_numeric(enriched["mid"], errors="coerce")
    bid = pd.to_numeric(enriched["bid"], errors="coerce")
    ask = pd.to_numeric(enriched["ask"], errors="coerce")

    checks = pd.DataFrame(index=enriched.index)
    checks["positive_expiry"] = expiry.gt(0)
    checks["positive_strike"] = strike.gt(0)
    checks["nonnegative_mid"] = mid.ge(0)
    checks["ordered_bid_ask"] = ask.ge(bid)
    checks["positive_iv"] = iv.gt(0) & np.isfinite(iv)
    checks["nonnegative_vega"] = vega.ge(0) & np.isfinite(vega)
    checks["right_ok"] = enriched["right"].isin(["call", "put"])
    checks["expiry_window"] = enriched["expiry_days"].between(MIN_EXPIRY_DAYS, MAX_EXPIRY_DAYS, inclusive="both")
    checks["moneyness_window"] = enriched["abs_log_moneyness"].le(MAX_ABS_LOG_MONEYNESS)
    checks["spread_window"] = enriched["relative_spread"].le(MAX_RELATIVE_SPREAD) | enriched["relative_spread"].isna()

    # Temporary workaround notebook only:
    # pandas comparisons can produce nullable boolean values when input columns contain NA.
    # Treat unknown check outcomes as failed checks, so reject-reason construction is stable.
    checks = checks.fillna(False).astype(bool)

    keep = checks.all(axis=1)
    selected = enriched.loc[keep].copy().reset_index(drop=True)
    rejected = enriched.loc[~keep].copy()
    rejected["reject_reasons"] = [
        tuple(checks.columns[~checks.loc[idx].to_numpy(dtype=bool)])
        for idx in rejected.index
    ]

    if selected.empty:
        return selected, rejected.reset_index(drop=True)

    selected["_right_sort"] = selected["right"].map({"call": 0, "put": 1}).fillna(9).astype(int)
    selected["_vega_rank"] = -pd.to_numeric(selected["vega"], errors="coerce").fillna(0.0)
    selected["_spread_rank"] = pd.to_numeric(selected["relative_spread"], errors="coerce").fillna(np.inf)
    selected["_atm_rank"] = pd.to_numeric(selected["abs_log_moneyness"], errors="coerce").fillna(np.inf)
    selected["_mny_bucket"] = pd.cut(
        selected["log_moneyness"],
        bins=[-np.inf, -0.20, -0.10, -0.03, 0.03, 0.10, 0.20, np.inf],
        labels=False,
    ).astype("float")

    selected = selected.sort_values(
        ["expiry_years", "_right_sort", "_mny_bucket", "_spread_rank", "_atm_rank", "_vega_rank", "strike", "contract_symbol"],
        ascending=[True, True, True, True, True, True, True, True],
    )

    selected = (
        selected.groupby(["expiry_years", "right"], group_keys=False, observed=False)
        .head(MAX_QUOTES_PER_EXPIRY_RIGHT)
        .reset_index(drop=True)
    )

    if len(selected) > MAX_QUOTES_TOTAL:
        selected = selected.sort_values(
            ["_spread_rank", "_atm_rank", "_vega_rank", "expiry_years", "right", "strike"],
            ascending=[True, True, True, True, True, True],
        ).head(MAX_QUOTES_TOTAL).reset_index(drop=True)

    selected = selected.drop(columns=[c for c in selected.columns if c.startswith("_")], errors="ignore")
    return selected.reset_index(drop=True), rejected.reset_index(drop=True)


def to_heston_quote_set(prepared_frame: pd.DataFrame, market_data):
    # Drop notebook-only helper columns before using the library converter.
    return heston_quote_set_from_frame(
        prepared_frame.drop(
            columns=["forward", "expiry_days", "log_moneyness", "abs_log_moneyness", "relative_spread"],
            errors="ignore",
        ),
        market_data,
    )


In [6]:
inventory_rows = []
prepared_by_underlying: dict[str, dict[str, Any]] = {}

for bundle in bundles:
    loaded = load_bundle(bundle)
    prepared, rejected = prepare_heston_ready_frame(loaded["heston_quotes"], loaded["market_data"])
    if prepared.empty:
        preflight = None
        recommendation = "empty_after_filter"
        messages = ""
        qset = None
    else:
        qset = to_heston_quote_set(prepared, loaded["market_data"])
        preflight = preflight_heston_quotes(qset)
        recommendation = preflight.recommendation
        messages = " | ".join(preflight.messages)

    call_count = int((prepared["right"] == "call").sum()) if not prepared.empty else 0
    put_count = int((prepared["right"] == "put").sum()) if not prepared.empty else 0
    expiry_count = int(prepared["expiry_years"].nunique()) if not prepared.empty else 0

    eligible = (
        recommendation == "ok"
        and len(prepared) >= MIN_QUOTES
        and expiry_count >= MIN_EXPIRIES
        and call_count >= MIN_QUOTES_PER_RIGHT
        and put_count >= MIN_QUOTES_PER_RIGHT
    )

    inventory_rows.append({
        "underlying": bundle.underlying,
        "date": bundle.date,
        "run_id": bundle.run_id,
        "root": str(bundle.root),
        "raw_quotes": int(len(loaded["heston_quotes"])),
        "selected_quotes": int(len(prepared)),
        "rejected_or_not_selected": int(len(loaded["heston_quotes"]) - len(prepared)),
        "expiry_count": expiry_count,
        "call_count": call_count,
        "put_count": put_count,
        "min_expiry_days": float(prepared["expiry_years"].min() * 365.0) if not prepared.empty else np.nan,
        "max_expiry_days": float(prepared["expiry_years"].max() * 365.0) if not prepared.empty else np.nan,
        "preflight": recommendation,
        "preflight_messages": messages,
        "eligible": bool(eligible),
    })

    prepared_by_underlying[bundle.underlying] = {
        "bundle": bundle,
        "loaded": loaded,
        "prepared": prepared,
        "rejected": rejected,
        "quote_set": qset,
    }

inventory = pd.DataFrame(inventory_rows).sort_values(["underlying", "date"]).reset_index(drop=True)
inventory


,underlying,date,run_id,root,raw_quotes,selected_quotes,rejected_or_not_selected,expiry_count,call_count,put_count,min_expiry_days,max_expiry_days,preflight,preflight_messages,eligible
0,IWM,2026-06-06,b4a-20260606T111054Z-d64ab9a6,C:\Users\ouwez\Documents\Quant\option-pricing-...,1925,900,1025,11,432,468,19.534094,166.534094,ok,All quotes passed Heston calibration preflight.,True
1,MSFT,2026-06-06,b4a-20260606T111126Z-0d4d8cc7,C:\Users\ouwez\Documents\Quant\option-pricing-...,1278,618,660,9,318,300,19.533717,166.533717,ok,All quotes passed Heston calibration preflight.,True
2,NVDA,2026-06-06,b4a-20260606T111142Z-82961440,C:\Users\ouwez\Documents\Quant\option-pricing-...,1596,470,1126,9,243,227,19.533537,166.533537,ok,All quotes passed Heston calibration preflight.,True
3,QQQ,2026-06-06,b4a-20260606T111029Z-2c8a4d1d,C:\Users\ouwez\Documents\Quant\option-pricing-...,4680,900,3780,12,593,307,19.534379,131.534379,ok,All quotes passed Heston calibration preflight.,True
4,SPY,2026-06-06,b4a-20260606T111002Z-a037b8fe,C:\Users\ouwez\Documents\Quant\option-pricing-...,5781,900,4881,15,287,613,19.534696,176.534696,ok,All quotes passed Heston calibration preflight.,True
5,TSLA,2026-06-06,b4a-20260606T111158Z-a2346800,C:\Users\ouwez\Documents\Quant\option-pricing-...,2513,900,1613,9,441,459,19.533346,166.533346,ok,All quotes passed Heston calibration preflight.,True


In [7]:
selected = (
    inventory[inventory["eligible"]]
    .sort_values(["underlying", "date", "selected_quotes", "run_id"], ascending=[True, False, False, True])
    .groupby("underlying", as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

if selected.empty:
    raise RuntimeError("No eligible Heston-ready bundles after temporary filtering.")

selected[[
    "underlying", "date", "run_id", "raw_quotes", "selected_quotes",
    "expiry_count", "call_count", "put_count", "min_expiry_days", "max_expiry_days", "preflight",
]]


,underlying,date,run_id,raw_quotes,selected_quotes,expiry_count,call_count,put_count,min_expiry_days,max_expiry_days,preflight
0,IWM,2026-06-06,b4a-20260606T111054Z-d64ab9a6,1925,900,11,432,468,19.534094,166.534094,ok
1,MSFT,2026-06-06,b4a-20260606T111126Z-0d4d8cc7,1278,618,9,318,300,19.533717,166.533717,ok
2,NVDA,2026-06-06,b4a-20260606T111142Z-82961440,1596,470,9,243,227,19.533537,166.533537,ok
3,QQQ,2026-06-06,b4a-20260606T111029Z-2c8a4d1d,4680,900,12,593,307,19.534379,131.534379,ok
4,SPY,2026-06-06,b4a-20260606T111002Z-a037b8fe,5781,900,15,287,613,19.534696,176.534696,ok
5,TSLA,2026-06-06,b4a-20260606T111158Z-a2346800,2513,900,9,441,459,19.533346,166.533346,ok


In [8]:
def tau_for_comparison(tau: float) -> float:
    if ROUND_LIBRARY_TAU_TO_QL_DAYS:
        return max(1, int(round(float(tau) * 365.0))) / 365.0
    return float(tau)


def price_with_library(quote_set, params: HestonParams) -> np.ndarray:
    prices = np.empty(quote_set.n_quotes, dtype=float)
    for tau_raw in np.unique(quote_set.expiry):
        tau = tau_for_comparison(float(tau_raw))
        idx_tau = np.flatnonzero(quote_set.expiry == tau_raw)
        for is_call_value, kind in ((True, OptionType.CALL), (False, OptionType.PUT)):
            idx = idx_tau[quote_set.is_call[idx_tau] == is_call_value]
            if idx.size == 0:
                continue
            prices[idx] = np.asarray(
                heston_price_from_ctx(
                    kind=kind,
                    strike=quote_set.strike[idx],
                    tau=tau,
                    ctx=quote_set.ctx,
                    params=params,
                    backend="gauss_legendre",
                    quad_cfg=ROBUST_QUAD_CFG,
                ),
                dtype=float,
            )
    return prices


def ql_flat_forward(evaluation_date, rate: float, day_counter):
    curve = ql.FlatForward(evaluation_date, float(rate), day_counter, ql.Continuous, ql.NoFrequency)
    return ql.YieldTermStructureHandle(curve)


def price_with_quantlib(quote_set, params: HestonParams, market_data) -> np.ndarray:
    if not QUANTLIB_AVAILABLE:
        raise RuntimeError("QuantLib is not installed.")

    day_counter = ql.Actual365Fixed()
    evaluation_date = ql.Date(2, ql.January, 2020)
    ql.Settings.instance().evaluationDate = evaluation_date

    risk_free_ts = ql_flat_forward(evaluation_date, float(market_data.rate), day_counter)
    dividend_ts = ql_flat_forward(evaluation_date, float(market_data.dividend_yield), day_counter)
    spot = ql.QuoteHandle(ql.SimpleQuote(float(market_data.spot)))

    process = ql.HestonProcess(
        risk_free_ts,
        dividend_ts,
        spot,
        float(params.v),
        float(params.kappa),
        float(params.vbar),
        float(params.eta),
        float(params.rho),
    )
    model = ql.HestonModel(process)
    engine = ql.AnalyticHestonEngine(model, int(QL_ENGINE_ORDER))

    out = np.empty(quote_set.n_quotes, dtype=float)
    for i in range(quote_set.n_quotes):
        maturity_days = max(1, int(round(float(quote_set.expiry[i]) * 365.0)))
        maturity_date = evaluation_date + maturity_days
        option_type = ql.Option.Call if bool(quote_set.is_call[i]) else ql.Option.Put
        option = ql.VanillaOption(
            ql.PlainVanillaPayoff(option_type, float(quote_set.strike[i])),
            ql.EuropeanExercise(maturity_date),
        )
        option.setPricingEngine(engine)
        out[i] = float(option.NPV())
    return out


def metrics(residual: np.ndarray) -> dict[str, float]:
    x = np.asarray(residual, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {"rmse": np.nan, "mae": np.nan, "max_abs": np.nan}
    return {
        "rmse": float(np.sqrt(np.mean(x * x))),
        "mae": float(np.mean(np.abs(x))),
        "max_abs": float(np.max(np.abs(x))),
    }


def comparison_metrics(market: np.ndarray, own: np.ndarray, ql_prices: np.ndarray) -> dict[str, Any]:
    own_market = metrics(own - market)
    ql_market = metrics(ql_prices - market)
    own_ql = metrics(own - ql_prices)
    scale = max(1.0, own_market["rmse"])

    return {
        "own_market_rmse": own_market["rmse"],
        "own_market_mae": own_market["mae"],
        "own_market_max_abs": own_market["max_abs"],
        "ql_market_rmse": ql_market["rmse"],
        "ql_market_mae": ql_market["mae"],
        "ql_market_max_abs": ql_market["max_abs"],
        "own_ql_rmse": own_ql["rmse"],
        "own_ql_mae": own_ql["mae"],
        "own_ql_max_abs": own_ql["max_abs"],
        "own_ql_rmse_as_frac_of_market_rmse": own_ql["rmse"] / scale,
        "own_ql_max_as_frac_of_market_rmse": own_ql["max_abs"] / scale,
        "implementation_flag": bool(
            own_ql["rmse"] > IMPLEMENTATION_RMSE_TOL
            or own_ql["max_abs"] > IMPLEMENTATION_MAX_ABS_TOL
        ),
    }


In [9]:
comparison_rows = []
fit_by_underlying = {}

for rec in selected.to_dict(orient="records"):
    underlying = rec["underlying"]
    item = prepared_by_underlying[underlying]
    loaded = item["loaded"]
    quote_set = item["quote_set"]
    market_data = loaded["market_data"]

    base = {
        "underlying": underlying,
        "date": rec["date"],
        "run_id": rec["run_id"],
        "status": "failed",
        "selected_quotes": int(rec["selected_quotes"]),
        "expiry_count": int(rec["expiry_count"]),
        "kappa": np.nan,
        "vbar": np.nan,
        "eta": np.nan,
        "rho": np.nan,
        "v": np.nan,
        "feller_ratio": np.nan,
        "warning_labels": "",
        "verdict": "",
        "message": "",
    }

    try:
        fit = calibrate_heston_multistart(
            quote_set,
            objective_type=OBJECTIVE_TYPE,
            bounds=HestonCalibrationBounds(),
            max_seeds=MAX_SEEDS,
            max_nfev=MAX_NFEV,
            use_analytic_jac=USE_ANALYTIC_JAC,
        )
        fit_by_underlying[underlying] = fit

        own_prices = price_with_library(quote_set, fit.best_params)
        if QUANTLIB_AVAILABLE:
            ql_prices = price_with_quantlib(quote_set, fit.best_params, market_data)
            ql_metrics = comparison_metrics(quote_set.mid, own_prices, ql_prices)
        else:
            own_market = metrics(own_prices - quote_set.mid)
            ql_metrics = {
                "own_market_rmse": own_market["rmse"],
                "own_market_mae": own_market["mae"],
                "own_market_max_abs": own_market["max_abs"],
                "ql_market_rmse": np.nan,
                "ql_market_mae": np.nan,
                "ql_market_max_abs": np.nan,
                "own_ql_rmse": np.nan,
                "own_ql_mae": np.nan,
                "own_ql_max_abs": np.nan,
                "own_ql_rmse_as_frac_of_market_rmse": np.nan,
                "own_ql_max_as_frac_of_market_rmse": np.nan,
                "implementation_flag": None,
            }

        report = run_heston_calibration_fit_diagnostics(
            quotes=quote_set,
            fit=fit,
            objective_type=OBJECTIVE_TYPE,
            bounds=HestonCalibrationBounds(),
            quad_cfg=ROBUST_QUAD_CFG,
            include_kappa_profile=False,
            objective_slice_grid_size=3,
        )
        summary = heston_calibration_fit_summary(report)
        params = fit.best_params

        base.update({
            "status": "ok",
            "success_count": int(fit.success_count),
            "failure_count": int(fit.failure_count),
            "best_cost": float(fit.best_run.cost),
            "kappa": float(params.kappa),
            "vbar": float(params.vbar),
            "eta": float(params.eta),
            "rho": float(params.rho),
            "v": float(params.v),
            "feller_ratio": summary.get("feller_ratio"),
            "warning_labels": ",".join(summary.get("warning_labels", [])),
            "verdict": summary.get("verdict"),
            **ql_metrics,
        })
    except Exception as exc:
        base["message"] = f"{type(exc).__name__}: {exc}"

    comparison_rows.append(base)

baseline_comparison = pd.DataFrame(comparison_rows).sort_values("underlying").reset_index(drop=True)
baseline_comparison


,underlying,date,run_id,status,selected_quotes,expiry_count,kappa,vbar,eta,rho,...,own_market_max_abs,ql_market_rmse,ql_market_mae,ql_market_max_abs,own_ql_rmse,own_ql_mae,own_ql_max_abs,own_ql_rmse_as_frac_of_market_rmse,own_ql_max_as_frac_of_market_rmse,implementation_flag
0,IWM,2026-06-06,b4a-20260606T111054Z-d64ab9a6,ok,900,11,2.876976,0.100420,1.926428,-0.247513,...,4.901666,0.736621,0.441425,4.901666,1.373628e-06,7.876994e-07,4.912833e-06,1.373628e-06,4.912833e-06,False
1,MSFT,2026-06-06,b4a-20260606T111126Z-0d4d8cc7,ok,618,9,15.511890,0.140168,2.846141,0.124524,...,9.855313,2.279059,1.746089,9.855313,1.667881e-09,3.325720e-10,1.300874e-08,7.318289e-10,5.707942e-09,False
2,NVDA,2026-06-06,b4a-20260606T111142Z-82961440,ok,470,9,8.318424,0.260560,2.654913,-0.002437,...,2.944082,0.761708,0.601982,2.944082,7.270827e-14,3.563008e-14,3.726186e-13,7.270827e-14,3.726186e-13,False
3,QQQ,2026-06-06,b4a-20260606T111029Z-2c8a4d1d,ok,900,12,5.373723,0.152607,5.000000,-0.626405,...,9.247291,2.873182,2.326153,9.247291,1.232916e-04,6.308379e-05,4.468141e-04,4.291115e-05,1.555119e-04,False
4,SPY,2026-06-06,b4a-20260606T111002Z-a037b8fe,ok,900,15,2.906510,0.066425,1.283836,-0.670248,...,16.571628,1.755620,0.694445,16.571628,4.343226e-04,1.902558e-04,2.123943e-03,2.473898e-04,1.209796e-03,False
5,TSLA,2026-06-06,b4a-20260606T111158Z-a2346800,ok,900,9,14.133160,0.294524,5.000000,0.034378,...,5.464966,1.210155,0.922656,5.464966,1.509776e-13,7.768755e-14,7.958079e-13,1.247589e-13,6.576085e-13,False


In [10]:
display_cols = [
    "underlying", "status", "selected_quotes", "expiry_count",
    "kappa", "eta", "rho", "feller_ratio",
    "own_market_rmse", "ql_market_rmse", "own_ql_rmse", "own_ql_max_abs",
    "own_ql_rmse_as_frac_of_market_rmse", "implementation_flag",
    "warning_labels", "verdict", "message",
]
display(baseline_comparison[[c for c in display_cols if c in baseline_comparison.columns]])

if not QUANTLIB_AVAILABLE:
    print("QuantLib is not installed, so only this library's calibration ran.")
else:
    ok = baseline_comparison[baseline_comparison["status"].eq("ok")].copy()
    if ok.empty:
        print("No successful calibration rows; QuantLib comparison did not run.")
    else:
        flagged = ok[ok["implementation_flag"].eq(True)]
        if flagged.empty:
            print("PASS: no same-parameter QuantLib implementation flags under the configured tolerances.")
        else:
            print("REVIEW: same-parameter QuantLib discrepancies exceeded tolerance.")
            display(flagged[["underlying", "own_ql_rmse", "own_ql_max_abs", "own_market_rmse", "implementation_flag"]])


,underlying,status,selected_quotes,expiry_count,kappa,eta,rho,feller_ratio,own_market_rmse,ql_market_rmse,own_ql_rmse,own_ql_max_abs,own_ql_rmse_as_frac_of_market_rmse,implementation_flag,warning_labels,verdict,message
0,IWM,ok,900,11,2.876976,1.926428,-0.247513,0.155697,0.736621,0.736621,1.373628e-06,4.912833e-06,1.373628e-06,False,FELLER_WEAK_OR_VIOLATED,usable_with_caution,
1,MSFT,ok,618,9,15.511890,2.846141,0.124524,0.536823,2.279059,2.279059,1.667881e-09,1.300874e-08,7.318289e-10,False,FELLER_WEAK_OR_VIOLATED,usable_with_caution,
2,NVDA,ok,470,9,8.318424,2.654913,-0.002437,0.615005,0.761708,0.761708,7.270827e-14,3.726186e-13,7.270827e-14,False,FELLER_WEAK_OR_VIOLATED,usable_with_caution,
3,QQQ,ok,900,12,5.373723,5.000000,-0.626405,0.065605,2.873182,2.873182,1.232916e-04,4.468141e-04,4.291115e-05,False,"PARAMETER_NEAR_BOUND,ETA_HIGH,FELLER_WEAK_OR_V...",usable_with_caution,
4,SPY,ok,900,15,2.906510,1.283836,-0.670248,0.234267,1.755621,1.755620,4.343226e-04,2.123943e-03,2.473898e-04,False,FELLER_WEAK_OR_VIOLATED,usable_with_caution,
5,TSLA,ok,900,9,14.133160,5.000000,0.034378,0.333004,1.210155,1.210155,1.509776e-13,7.958079e-13,1.247589e-13,False,"PARAMETER_NEAR_BOUND,ETA_HIGH,FELLER_WEAK_OR_V...",usable_with_caution,


PASS: no same-parameter QuantLib implementation flags under the configured tolerances.


In [11]:
def pricing_disagreement_detail(underlying: str) -> pd.DataFrame:
    item = prepared_by_underlying[underlying]
    quote_set = item["quote_set"]
    market_data = item["loaded"]["market_data"]
    prepared = item["prepared"].reset_index(drop=True)
    fit = fit_by_underlying[underlying]
    params = fit.best_params

    own_prices = price_with_library(quote_set, params)
    ql_prices = price_with_quantlib(quote_set, params, market_data)

    detail = prepared.copy()
    detail["own_model_price"] = own_prices
    detail["ql_model_price"] = ql_prices
    detail["own_minus_ql"] = own_prices - ql_prices
    detail["own_minus_market"] = own_prices - quote_set.mid
    detail["ql_minus_market"] = ql_prices - quote_set.mid
    detail["abs_own_minus_ql"] = np.abs(detail["own_minus_ql"])
    return detail.sort_values("abs_own_minus_ql", ascending=False)


if QUANTLIB_AVAILABLE and not baseline_comparison.empty:
    ok = baseline_comparison[baseline_comparison["status"].eq("ok")].dropna(subset=["own_ql_max_abs"])
    if ok.empty:
        print("Skipping detail: no successful QuantLib comparison rows.")
    else:
        for underlying in ok.sort_values("own_ql_max_abs", ascending=False)["underlying"].head(3):
            print(f"\nLargest same-parameter pricing discrepancies for {underlying}")
            detail = pricing_disagreement_detail(underlying)
            cols = [
                "contract_symbol", "expiry_days", "strike", "right", "mid",
                "own_model_price", "ql_model_price", "own_minus_ql",
                "own_minus_market", "ql_minus_market", "log_moneyness",
            ]
            display(detail[[c for c in cols if c in detail.columns]].head(12))
else:
    print("Skipping detail: QuantLib unavailable or no comparison table.")



Largest same-parameter pricing discrepancies for SPY


,contract_symbol,expiry_days,strike,right,mid,own_model_price,ql_model_price,own_minus_ql,own_minus_market,ql_minus_market,log_moneyness
345,SPY260626P00710000,19.534696,710.0,put,4.605,4.117464,4.115340,0.002124,-0.487536,-0.489660,-0.039815
876,SPY260626P00701000,19.534696,701.0,put,3.495,2.973749,2.975861,-0.002112,-0.521251,-0.519139,-0.052572
539,SPY260626P00685000,19.534696,685.0,put,2.27,1.644914,1.647009,-0.002095,-0.625086,-0.622991,-0.075661
229,SPY260626P00718000,19.534696,718.0,put,5.945,5.462724,5.464816,-0.002093,-0.482276,-0.480184,-0.028610
889,SPY260626P00719000,19.534696,719.0,put,6.195,5.658478,5.660555,-0.002077,-0.536522,-0.534445,-0.027218
86,SPY260626P00702000,19.534696,702.0,put,3.615,3.083845,3.085908,-0.002063,-0.531155,-0.529092,-0.051146
541,SPY260626P00736000,19.534696,736.0,put,11.33,10.245006,10.247066,-0.002060,-1.084994,-1.082934,-0.003850
655,SPY260626C00745000,19.534696,745.0,call,7.57,7.932808,7.930808,0.002001,0.362808,0.360808,0.008304
798,SPY260626C00754000,19.534696,754.0,call,3.855,4.172304,4.174265,-0.001961,0.317304,0.319265,0.020313
756,SPY260626P00711000,19.534696,711.0,put,4.85,4.266722,4.264785,0.001937,-0.583278,-0.585215,-0.038407



Largest same-parameter pricing discrepancies for QQQ


,contract_symbol,expiry_days,strike,right,mid,own_model_price,ql_model_price,own_minus_ql,own_minus_market,ql_minus_market,log_moneyness
438,QQQ260626P00676000,19.534379,676.0,put,8.545,12.050943,12.050496,0.000447,3.505943,3.505496,-0.037140
151,QQQ260626P00684000,19.534379,684.0,put,10.555,13.696539,13.696986,-0.000446,3.141539,3.141986,-0.025375
496,QQQ260626C00668000,19.534379,668.0,call,45.21,44.195682,44.196127,-0.000445,-1.014318,-1.013873,-0.049045
537,QQQ260626P00660000,19.534379,660.0,put,5.595,9.446566,9.446122,0.000445,3.851566,3.851122,-0.061093
672,QQQ260626C00660000,19.534379,660.0,call,52.0,50.974514,50.974069,0.000445,-1.025486,-1.025931,-0.061093
553,QQQ260626P00692000,19.534379,692.0,put,12.915,15.675651,15.675212,0.000439,2.760651,2.760212,-0.013747
284,QQQ260626P00675000,19.534379,675.0,put,8.39,11.863917,11.863491,0.000426,3.473917,3.473491,-0.038620
400,QQQ260626C00709000,19.534379,709.0,call,16.415,14.263995,14.263570,0.000425,-2.151005,-2.151430,0.010523
145,QQQ260626C00718000,19.534379,718.0,call,12.105,9.935694,9.936118,-0.000424,-2.169306,-2.168882,0.023137
452,QQQ260626C00585000,19.534379,585.0,call,120.47,119.516650,119.516227,0.000423,-0.953350,-0.953773,-0.181721



Largest same-parameter pricing discrepancies for IWM


,contract_symbol,expiry_days,strike,right,mid,own_model_price,ql_model_price,own_minus_ql,own_minus_market,ql_minus_market,log_moneyness
842,IWM260626P00302000,19.534094,302.0,put,21.28,21.650272,21.650267,0.000005,0.370272,0.370267,0.071078
664,IWM260626P00295000,19.534094,295.0,put,15.365,15.545961,15.545956,0.000005,0.180961,0.180956,0.047626
680,IWM260626P00271000,19.534094,271.0,put,3.175,2.961819,2.961823,-0.000005,-0.213181,-0.213177,-0.037230
854,IWM260626P00258000,19.534094,258.0,put,1.18,1.075190,1.075194,-0.000005,-0.104810,-0.104806,-0.086389
324,IWM260626P00277500,19.534094,277.5,put,5.115,4.837247,4.837251,-0.000005,-0.277753,-0.277749,-0.013528
336,IWM260626C00277500,19.534094,277.5,call,8.975,8.622308,8.622312,-0.000005,-0.352692,-0.352688,-0.013528
456,IWM260626C00288000,19.534094,288.0,call,3.415,3.555107,3.555103,0.000005,0.140107,0.140103,0.023612
164,IWM260626P00288000,19.534094,288.0,put,10.04,10.249044,10.249039,0.000005,0.209044,0.209039,0.023612
41,IWM260626C00281000,19.534094,281.0,call,6.8,6.563197,6.563193,0.000005,-0.236803,-0.236807,-0.000994
105,IWM260626P00281000,19.534094,281.0,put,6.31,6.271135,6.271131,0.000005,-0.038865,-0.038869,-0.000994


In [12]:
def conclusion_from_quantlib_table(table: pd.DataFrame) -> str:
    if table.empty:
        return "No comparison rows were produced."
    if not QUANTLIB_AVAILABLE:
        return "QuantLib was not installed, so the independent cross-check could not run."

    ok = table[table["status"].eq("ok")].copy()
    if ok.empty:
        return "No successful Heston calibration rows were available for QuantLib comparison."

    max_rmse = float(ok["own_ql_rmse"].max(skipna=True))
    max_abs = float(ok["own_ql_max_abs"].max(skipna=True))
    max_frac = float(ok["own_ql_rmse_as_frac_of_market_rmse"].max(skipna=True))
    flagged = ok[ok["implementation_flag"].eq(True)]

    lines = [
        "QuantLib same-parameter pricing cross-check:",
        f"- successful rows: {len(ok)} / {len(table)}",
        f"- max own-vs-QuantLib RMSE: {max_rmse:.6g}",
        f"- max own-vs-QuantLib absolute error: {max_abs:.6g}",
        f"- max own-vs-QuantLib RMSE as fraction of own market RMSE: {max_frac:.6g}",
    ]

    if flagged.empty:
        lines.append(
            "- No implementation flags were triggered. This supports interpreting the earlier "
            "Heston issues as model/calibration limitations rather than a pricing implementation bug."
        )
    else:
        names = ", ".join(flagged["underlying"].astype(str))
        lines.append(
            f"- REVIEW required for: {names}. Same-parameter QuantLib disagreement exceeded tolerance."
        )
    return "\n".join(lines)


print(conclusion_from_quantlib_table(baseline_comparison))


QuantLib same-parameter pricing cross-check:
- successful rows: 6 / 6
- max own-vs-QuantLib RMSE: 0.000434323
- max own-vs-QuantLib absolute error: 0.00212394
- max own-vs-QuantLib RMSE as fraction of own market RMSE: 0.00024739
- No implementation flags were triggered. This supports interpreting the earlier Heston issues as model/calibration limitations rather than a pricing implementation bug.
